### DS4420 Project: Indoor Scene Recognition
Colin Chu and Ethan Fang

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Dense, Input
from keras.layers import Conv2D, MaxPooling2D, Flatten
from keras.optimizers import SGD
from keras.losses import binary_crossentropy
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from PIL import Image, ImageOps
import os
def folder_to_numpy(folder_path, target_width=None, target_height=None):
  images = []

  for image in os.listdir(folder_path):
    with Image.open(os.path.join(folder_path, image)) as img:
        grayscale_img = img.convert('L')

        if target_width and target_height:
            padded_img = ImageOps.pad(grayscale_img, (target_width, target_height), color=0)
        else:
            padded_img = grayscale_img

        grayscale_array = np.array(padded_img)
        images.append(grayscale_array)

  return np.array(images)

In [ ]:
images_dir = '/content/drive/MyDrive/ds4420/project/Images'
# images_dir = '/content/drive/MyDrive/project/Images'

scene_folders = os.listdir(images_dir)
all_data = []
all_labels = []

for label, folder_name in enumerate(scene_folders):
    folder_path = os.path.join(images_dir, folder_name)
    data = folder_to_numpy(folder_path, 499, 540)
    labels = np.full(data.shape[0], label)
    all_data.append(data)
    all_labels.append(labels)

indoor_data = np.concatenate(all_data, axis=0)
y = np.concatenate(all_labels, axis=0)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(indoor_data, y, test_size=0.3, random_state=1, stratify=y)

x_train = x_train / 255.0
x_test = x_test / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

In [ ]:
inpx = Input(shape=(540, 499, 1))

con_layer = Conv2D(1, kernel_size=(4, 4), strides=2, activation=None, padding='same')(inpx)
pool_layer = MaxPooling2D(pool_size=(2, 2))(con_layer)
flat_G = Flatten()(pool_layer)
hid_layer = Dense(250, activation='relu')(flat_G)
hid_layer2 = Dense(100, activation='tanh')(hid_layer)
out_layer = Dense(1, activation='sigmoid')(hid_layer2)

In [ ]:
model = Model([inpx], out_layer)
model.compile(optimizer=SGD(), loss=binary_crossentropy, metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, batch_size=16, epochs=10)

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('loss=', score[0])
print('accuracy=', score[1])

In [ ]:
preds = model.predict(x_test)
y_true = np.asarray(y_test).ravel().astype(int)
y_pred = (preds.reshape(-1) >= 0.5).astype(int)
tbl = pd.crosstab(y_pred, y_true, rownames=['Predicted'], colnames=['Actual'])
print(tbl)

In [ ]:
conv_weights = model.layers[1].get_weights()[0]
W = conv_weights[:,:,0,0]
print("Convolutional kernel weights:")
print(W)

In [ ]:
from PIL import Image, ImageOps
def convert_image_to_grayscale_with_padding(image_path, target_width=None, target_height=None):
    """
    Converts a single .jpg image to grayscale and optionally pads it to the target dimensions.

    Args:
        image_path (str): Path to the input .jpg image.
        target_width (int, optional): Target width for padding. If None, keeps the original width.
        target_height (int, optional): Target height for padding. If None, keeps the original height.

    Returns:
        np.ndarray: Grayscale image as a numpy array, with optional padding applied.
    """
    # Open the image
    with Image.open(image_path) as img:
        # Convert to grayscale
        grayscale_img = img.convert('L')

        # If target dimensions are provided, pad the image
        if target_width and target_height:
            padded_img = ImageOps.pad(grayscale_img, (target_width, target_height), color=0)
        else:
            padded_img = grayscale_img

        # Convert the image to a numpy array
        grayscale_array = np.array(padded_img)

    return grayscale_array

X = convert_image_to_grayscale_with_padding('cat.jpg', 499, 540)
X

In [ ]:
X = X[:, :, np.newaxis]
X = np.expand_dims(X, axis=0)
model.predict(X)

In [ ]:
def plot_img(x, im_shape):
    plt.imshow(x.reshape(im_shape), cmap='gray')
    plt.xticks([])
    plt.yticks([])
    plt.gcf().set_size_inches(4, 4)

In [ ]:
def convMat(X, W):
    k_h, k_w = W.shape
    p_h, p_w = X.shape

    q_h = p_h - k_h + 1
    q_w = p_w - k_w + 1
    G = np.zeros((q_h, q_w))

    for m in range(q_h):
        for n in range(q_w):
            submatrix = X[m:m+k_h, n:n+k_w]
            G[m, n] = np.sum(W * submatrix)

    return G

G = convMat(X[0][:,:,0], W)

plot_img(G, G.shape)